# 🏢 CORP-ENV · Qwen 2.5-7B-Instruct — SFT + RLVR Training

**End-to-end reproducible notebook** for training a Qwen 2.5-7B-Instruct agent on CORP-ENV using Supervised Fine-Tuning (SFT) followed by Rejection-Sampling RL on Verifiable Rewards (RLVR).

CORP-ENV is a multi-agent corporate decision environment where a Master Agent governs a **Shared Workspace Document (SWD)** across long-horizon planning episodes, coordinating frozen worker agents. Rewards measure SWD integrity, task completion, milestone adherence, reasoning density, and LLM-judge scores.

| Component | Detail |
|---|---|
| **Base model** | `Qwen/Qwen2.5-7B-Instruct` |
| **SFT script** | `training/train_sft.py` |
| **RLVR script** | `training/train_rlvr.py` |
| **Tasks** | E1 Launch Readiness, M1 Budget Reallocation, H1 Acquisition Defence |
| **Runtime** | Colab GPU / Lightning AI H100 / Any CUDA GPU |

---

## 1️⃣ Setup & Installation

In [ ]:
import os

# ===== Configuration =====
REPO_URL = "https://huggingface.co/spaces/Navigam1108/corp-env"  # Change to your repo
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
HF_ORG_OR_USER = "Navigam1108"  # Your HF username/org

# SFT hyperparameters
SFT_MAX_STEPS = 30        # Quick judge smoke; set -1 for full-epoch training
SFT_EPOCHS = 2.0
SFT_LR = 2e-4
SFT_BATCH_SIZE = 1
SFT_GRAD_ACCUM = 8

# RLVR hyperparameters
RLVR_ROUNDS = 3
RLVR_MAX_PROMPTS = 128
RLVR_N_SAMPLES = 8
RLVR_TEMPERATURE = 0.7

# Eval
EVAL_EPISODES = 3
EVAL_MAX_STEPS = 30

In [ ]:
# Clone and install
!git clone {REPO_URL} corp_gym 2>/dev/null || echo 'Repo already cloned'
%cd corp_gym
!pip install -U pip
!pip install -e ".[training,plots]"

## 2️⃣ Hugging Face Login (optional)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3️⃣ Environment Validation

Quick sanity check: tests pass and `openenv validate` succeeds.

In [ ]:
!python -m pytest tests/ -v --tb=short 2>/dev/null || echo 'No tests found or tests skipped'
!openenv validate

## 4️⃣ Data Preparation

Verify raw trajectory examples through the real `CorpEnvironment`, filter bad trajectories, and prepare the SFT dataset.

In [ ]:
# Import and verify E1/M1 examples
!python scripts/verify_examples.py \
  --input data/raw/e1_m1_examples.jsonl \
  --clean data/processed/e1_m1_clean.jsonl \
  --rejected data/processed/e1_m1_rejected.jsonl \
  --summary results/e1_m1_verification_summary.json \
  --all-records results/e1_m1_verification_all.jsonl

# Generate and verify H1 seed data
!python scripts/generate_sft_data.py \
  --tasks h1_acquisition_defence --per-task 24 \
  --output data/raw/h1_seed.jsonl

!python scripts/verify_examples.py \
  --input data/raw/h1_seed.jsonl \
  --clean data/processed/h1_seed_clean.jsonl \
  --rejected data/processed/h1_seed_rejected.jsonl

# Merge into final SFT dataset
!python scripts/prepare_sft_data.py

In [ ]:
# Check data stats
import json
from pathlib import Path

sft_path = Path("data/sft/e1_m1_h1_examples.jsonl")
if sft_path.exists():
    lines = [json.loads(l) for l in sft_path.read_text().strip().splitlines() if l.strip()]
    print(f"\n✅ SFT dataset: {len(lines)} examples")
    # Count by number of messages
    turn_counts = [len(ex['messages']) for ex in lines]
    print(f"   Avg turns per example: {sum(turn_counts)/len(turn_counts):.1f}")
    print(f"   Min/Max turns: {min(turn_counts)} / {max(turn_counts)}")
else:
    print("❌ SFT dataset not found. Check data preparation above.")

## 5️⃣ Baseline Evaluation

Run scripted (weak) and oracle policies to establish comparison baselines.

In [ ]:
!python eval.py --policy scripted_weak --label baseline --episodes {EVAL_EPISODES}
!python eval.py --policy oracle --label oracle --episodes {EVAL_EPISODES}

## 6️⃣ SFT Training (Unsloth + TRL)

Fine-tune Qwen 2.5-7B-Instruct with LoRA using verified CORP-ENV trajectories.

- Uses `unsloth.FastLanguageModel` for 4-bit QLoRA
- Uses `trl.SFTTrainer` with messages-format conversational SFT
- LoRA `r=32`, targets all attention + MLP projections

In [ ]:
!python training/train_sft.py \
  --model {BASE_MODEL} \
  --data data/sft/e1_m1_h1_examples.jsonl \
  --output outputs/sft_adapter \
  --max-steps {SFT_MAX_STEPS} \
  --epochs {SFT_EPOCHS} \
  --lr {SFT_LR} \
  --batch-size {SFT_BATCH_SIZE} \
  --grad-accum {SFT_GRAD_ACCUM} \
  --push-to-hub {HF_ORG_OR_USER}/corp-env-sft-qwen2.5-7b

## 7️⃣ Evaluate SFT Adapter

In [ ]:
!python eval.py \
  --policy hf \
  --label sft \
  --model {BASE_MODEL} \
  --adapter outputs/sft_adapter \
  --episodes {EVAL_EPISODES} \
  --max-steps {EVAL_MAX_STEPS}

## 8️⃣ RLVR Training (Rejection-Sampling FT)

RLVR improves on SFT by:
1. Sampling N completions per prompt at non-zero temperature
2. Scoring each with the real `CorpEnvironment` verifier
3. Keeping the best completion per prompt above a reward threshold
4. SFT on that curated set
5. Repeating for multiple outer rounds

This avoids the zero-variance gradient problem seen with GRPO on CORP-ENV.

In [ ]:
!python training/train_rlvr.py \
  --model {BASE_MODEL} \
  --adapter outputs/sft_adapter \
  --examples data/processed/e1_m1_clean.jsonl,data/processed/h1_seed_clean.jsonl \
  --output outputs/rlvr_adapter \
  --rounds {RLVR_ROUNDS} \
  --n-samples {RLVR_N_SAMPLES} \
  --temperature {RLVR_TEMPERATURE} \
  --max-prompts {RLVR_MAX_PROMPTS} \
  --strict-json \
  --use-stub-workers \
  --disable-llm-judge \
  --stats-file results/runs/rlvr_qwen2.5_7b_stats.jsonl \
  --push-to-hub {HF_ORG_OR_USER}/corp-env-rlvr-qwen2.5-7b

## 9️⃣ Evaluate RLVR Adapter

In [ ]:
!python eval.py \
  --policy hf \
  --label rlvr \
  --model {BASE_MODEL} \
  --adapter outputs/rlvr_adapter \
  --episodes {EVAL_EPISODES} \
  --max-steps {EVAL_MAX_STEPS}

## 📊 Generate Comparison Plots

Aggregate all eval runs and generate:
- Terminal reward comparison (grouped bar chart)
- Verifier pass rate by task
- Invalid action rate
- Reward curve over episode steps

In [ ]:
!python plot_results.py \
  --inputs results/runs \
  --output-dir results/model_compare_qwen25_7b

In [ ]:
from IPython.display import Image, display, Markdown
from pathlib import Path

plot_dir = Path("results/model_compare_qwen25_7b")
if not plot_dir.exists():
    plot_dir = Path("results/model_compare_qwen25_fresh_no_grpo_ep5rlvr")

for png in sorted(plot_dir.glob("*.png")):
    display(Markdown(f"### {png.stem.replace('_', ' ').title()}"))
    display(Image(filename=str(png), width=800))

# Show summary table
summary_md = plot_dir / "comparison_summary.md"
if summary_md.exists():
    display(Markdown(summary_md.read_text()))

## 📋 Results Summary

Expected progression for Qwen 2.5-7B-Instruct on CORP-ENV:

| Stage | E1 Terminal Reward | M1 Terminal Reward | H1 Terminal Reward | M1 Success |
|-------|-------------------|-------------------|-------------------|------------|
| Baseline (weak) | 0.590 | 0.198 | 0.257 | 0% |
| Base (pre-trained) | 0.910 | 0.765 | 0.810 | 0% |
| SFT | 0.910 | 0.943 | 0.889 | 100% |
| RLVR | 0.910 | 0.932 | 0.779 | 80% |

> **Key takeaway**: SFT dramatically improves M1 (budget reallocation) from 0% to 100% success rate. RLVR maintains strong performance while reducing reliance on fixed trajectories.